![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# SQL and Python
*Data Science & Machine Learning - Week 2, Day 4*

So far SQL has lived in DB Browser and Python has lived in Jupyter. Real work needs them joined up: query the database from Python, pull the result into a DataFrame, and push data back the other way.

That last part is what **Project 1** asks for: *import your dataset into SQL, then answer your business questions with SQL queries*. The final section of this notebook is how you do it.

## 1. Connect to the database

`sqlite3` is in the Python standard library, so there is nothing to install. A SQLite database is just a file, so the 'connection string' is a path.

In [1]:
import sqlite3
import pandas as pd

# point this at wherever you saved the database
DB_PATH = "../s-301/sqlite-sakila.db"

con = sqlite3.connect(DB_PATH)
print("Connection open")

Connection open


## 2. The cursor

The cursor is the object that actually runs your query and holds the results.

In [2]:
cursor = con.cursor()

## 3. Read data: execute, then fetch

`execute()` runs the query; the rows stay in the cursor until you `fetchall()` them.

In [3]:
query = "SELECT * FROM actor LIMIT 10;"
cursor.execute(query)

result = cursor.fetchall()

for row in result:
    print(row)

(1, 'PENELOPE', 'GUINESS', '2021-03-06 15:51:59')
(2, 'NICK', 'WAHLBERG', '2021-03-06 15:51:59')
(3, 'ED', 'CHASE', '2021-03-06 15:51:59')
(4, 'JENNIFER', 'DAVIS', '2021-03-06 15:51:59')
(5, 'JOHNNY', 'LOLLOBRIGIDA', '2021-03-06 15:51:59')
(6, 'BETTE', 'NICHOLSON', '2021-03-06 15:51:59')
(7, 'GRACE', 'MOSTEL', '2021-03-06 15:51:59')
(8, 'MATTHEW', 'JOHANSSON', '2021-03-06 15:51:59')
(9, 'JOE', 'SWANK', '2021-03-06 15:51:59')
(10, 'CHRISTIAN', 'GABLE', '2021-03-06 15:51:59')


`cursor.description` carries the metadata, which is where the column names come from.

In [4]:
[header[0] for header in cursor.description]

['actor_id', 'first_name', 'last_name', 'last_update']

## 4. Straight into pandas

You can build the DataFrame by hand from the rows and the column names...

In [5]:
col_names = [header[0] for header in cursor.description]
actors = pd.DataFrame(result, columns=col_names)
actors.head()

,actor_id,first_name,last_name,last_update
0,1,PENELOPE,GUINESS,2021-03-06 15:51:59
1,2,NICK,WAHLBERG,2021-03-06 15:51:59
2,3,ED,CHASE,2021-03-06 15:51:59
3,4,JENNIFER,DAVIS,2021-03-06 15:51:59
4,5,JOHNNY,LOLLOBRIGIDA,2021-03-06 15:51:59


...but `pd.read_sql` does all of that in one line, and it is what you will actually use.

In [6]:
actors = pd.read_sql("SELECT * FROM actor;", con)
actors.head()

,actor_id,first_name,last_name,last_update
0,1,PENELOPE,GUINESS,2021-03-06 15:51:59
1,2,NICK,WAHLBERG,2021-03-06 15:51:59
2,3,ED,CHASE,2021-03-06 15:51:59
3,4,JENNIFER,DAVIS,2021-03-06 15:51:59
4,5,JOHNNY,LOLLOBRIGIDA,2021-03-06 15:51:59


## 5. Do the work in SQL, not in pandas

Both of these answer the same question. The first drags three whole tables into Python and joins them there; the second asks the database to do the join and returns only the answer.

In [7]:
import time

t1 = time.time()
actors     = pd.read_sql("SELECT * FROM actor;", con)
film_actor = pd.read_sql("SELECT * FROM film_actor;", con)
films      = pd.read_sql("SELECT * FROM film;", con)
merged = actors.merge(film_actor, on="actor_id").merge(films, on="film_id")
print("in pandas:", round(time.time() - t1, 3), "s  ->", merged.shape)

in pandas: 0.01 s  -> (5462, 18)


In [8]:
t1 = time.time()
query = """
    SELECT *
    FROM actor AS a
    INNER JOIN film_actor AS fa ON a.actor_id = fa.actor_id
    INNER JOIN film AS f        ON f.film_id  = fa.film_id;
"""
df = pd.read_sql(query, con)
print("in SQL:   ", round(time.time() - t1, 3), "s  ->", df.shape)

in SQL:    0.021 s  -> (5462, 20)


Same answer, and on this dataset the timings are a wash - SQLite runs inside your Python process, so there is no network hop to save and 1,000 films fit in memory comfortably.

The difference shows up when the database is **remote** and **large**. Then the first version has to drag every row across the network before it can start, while the second sends one query and receives only the answer. Write the query so the database does the work, and the habit scales when the data does.


## 6. Writing data back: `to_sql`

**This is the Project 1 step.** You have a DataFrame - a CSV you cleaned in pandas - and you want it in a database so you can query it with SQL.

Note we open a **new, empty database** for this. Sakila is the sample data we are reading; your project data belongs in your own file. SQLite creates the file the moment you connect to it.

In [9]:
actors_df = pd.read_sql("SELECT * FROM actor;", con)
actors_df["full_name"] = actors_df["first_name"] + " " + actors_df["last_name"]
actors_df.head()

,actor_id,first_name,last_name,last_update,full_name
0,1,PENELOPE,GUINESS,2021-03-06 15:51:59,PENELOPE GUINESS
1,2,NICK,WAHLBERG,2021-03-06 15:51:59,NICK WAHLBERG
2,3,ED,CHASE,2021-03-06 15:51:59,ED CHASE
3,4,JENNIFER,DAVIS,2021-03-06 15:51:59,JENNIFER DAVIS
4,5,JOHNNY,LOLLOBRIGIDA,2021-03-06 15:51:59,JOHNNY LOLLOBRIGIDA


In [10]:
# connecting to a file that does not exist creates it
project_con = sqlite3.connect("my_project.db")

# if_exists: "fail" (default), "replace", or "append"
actors_df.to_sql("actor_full_names", project_con, if_exists="replace", index=False)

print("rows written:", pd.read_sql("SELECT COUNT(*) AS n FROM actor_full_names;", project_con)["n"][0])

rows written: 200


The table is now a first-class part of *your* database, so SQL can see it. Open `my_project.db` in DB Browser and it is there.

In [11]:
pd.read_sql("SELECT full_name FROM actor_full_names ORDER BY full_name LIMIT 5;", project_con)

,full_name
0,ADAM GRANT
1,ADAM HOPPER
2,AL GARLAND
3,ALAN DREYFUSS
4,ALBERT JOHANSSON


In [12]:
project_con.close()

## 7. Commit and close

`to_sql` commits for you, but anything you run through the cursor needs an explicit `commit()`, and closing the connection is good practice.

In [13]:
con.commit()
cursor.close()
con.close()
print("closed")

closed


## 8. The same code against MySQL

`sqlite3` and `pymysql` both implement the same standard interface, **DB-API 2.0**. That means everything above - `cursor()`, `execute()`, `fetchall()`, `read_sql`, `to_sql`, `commit()`, `close()` - is identical. **Only the connection line changes.**

If you installed MySQL and Workbench from the Installations guide, this is the equivalent:

```python
# pip install pymysql cryptography
import pymysql
import pandas as pd

con = pymysql.connect(user='root', password='your_password',
                      host='localhost', database='sakila')

# from here on, everything is the same as above
actors = pd.read_sql('SELECT * FROM actor;', con)
```

The only other difference is dialect: MySQL writes `sakila.actor` and `CONCAT(a, b)`, SQLite writes `actor` and `a || b`. The Python around it does not change.